In [ ]:
# CELDA 1 — Instalar dependencias (ejecutar si no lo hiciste)
!pip install -q opencv-python-headless scikit-learn matplotlib seaborn

In [3]:
# ============================================================
# CELDA 3 — Verificación de estructura train/val/test
# ============================================================

import os

# ------------------------------------------------------------
# CONFIGURACIÓN
# ------------------------------------------------------------

DATA_DIR = "/mnt/d/Deteccion de enfermedades en maiz/imagenes_dataset_final"

SPLITS = ["train", "val", "test"]

VALID_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')

# ------------------------------------------------------------
# FUNCIÓN DE CONTEO
# ------------------------------------------------------------

def count_images(folder):
    count = 0

    if os.path.isdir(folder):
        for f in os.listdir(folder):
            if f.lower().endswith(VALID_EXTS):
                count += 1

    return count

# ------------------------------------------------------------
# VERIFICACIÓN
# ------------------------------------------------------------

print(f"🔍 Verificando estructura en:\n{DATA_DIR}\n")

if not os.path.exists(DATA_DIR):
    print("❌ ERROR: La carpeta principal no existe.")

else:

    total_general = 0

    for split in SPLITS:

        split_path = os.path.join(DATA_DIR, split)

        print("=" * 50)
        print(f"📂 CONJUNTO: {split.upper()}")
        print("=" * 50)

        if not os.path.exists(split_path):
            print(f"⚠️ No existe: {split_path}\n")
            continue

        classes = sorted([
            d for d in os.listdir(split_path)
            if os.path.isdir(os.path.join(split_path, d))
        ])

        print(f"{'CLASE':<20} | {'IMÁGENES':<10}")
        print("-" * 35)

        subtotal = 0

        for cls in classes:

            class_path = os.path.join(split_path, cls)

            num_images = count_images(class_path)

            print(f"{cls:<20} | {num_images:<10}")

            subtotal += num_images

        print("-" * 35)
        print(f"TOTAL {split.upper():<13} | {subtotal:<10}\n")

        total_general += subtotal

    print("=" * 50)
    print(f"📊 TOTAL GENERAL DEL DATASET: {total_general}")
    print("=" * 50)

🔍 Verificando estructura en:
/mnt/d/Deteccion de enfermedades en maiz/imagenes_dataset_final

📂 CONJUNTO: TRAIN
CLASE                | IMÁGENES  
-----------------------------------
roya_comun           | 650       
sana                 | 650       
tizon_foliar         | 650       
-----------------------------------
TOTAL TRAIN         | 1950      

📂 CONJUNTO: VAL
CLASE                | IMÁGENES  
-----------------------------------
roya_comun           | 97        
sana                 | 105       
tizon_foliar         | 97        
-----------------------------------
TOTAL VAL           | 299       

📂 CONJUNTO: TEST
CLASE                | IMÁGENES  
-----------------------------------
roya_comun           | 98        
sana                 | 105       
tizon_foliar         | 98        
-----------------------------------
TOTAL TEST          | 301       

📊 TOTAL GENERAL DEL DATASET: 2550


In [ ]:
import os
import cv2
import random
import re
from datetime import datetime
import albumentations as A
from tqdm import tqdm

# ==================================================
# CONFIGURACIÓN GENERAL
# ==================================================
DATASET_DIR = "imagenes_maiz"
TARGET_COUNT = 650
IMG_EXTS = (".jpg", ".jpeg", ".png")

CLASSES_TO_AUGMENT = ["tizon_foliar"]

# ==================================================
# REGEX FORMATO FINAL OBLIGATORIO
# IMG-YYYYMMDD-WAxxxxx.jpg
# ==================================================
IMG_PATTERN = re.compile(r"^IMG-\d{8}-WA\d+\.(jpg|jpeg|png)$", re.IGNORECASE)

# ==================================================
# PIPELINE MÍNIMO (SEGURO)
# ==================================================
augmenter = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5)
])

# ==================================================
# FUNCIÓN DE AUMENTO POR CLASE
# ==================================================
def augment_class(class_path):

    # --------------------------------------------------
    # 1. CONTAR TODAS LAS IMÁGENES REALES (SIN REGLAS)
    # --------------------------------------------------
    all_images = [
        f for f in os.listdir(class_path)
        if f.lower().endswith(IMG_EXTS)
    ]

    total_real = len(all_images)

    if total_real >= TARGET_COUNT:
        print(f"✅ {os.path.basename(class_path)} ya tiene {total_real} imágenes")
        return

    # --------------------------------------------------
    # 2. IMÁGENES FUENTE (SOLO FORMATO CORRECTO)
    # --------------------------------------------------
    source_images = [
        f for f in all_images
        if IMG_PATTERN.match(f)
    ]

    if not source_images:
        print(f"❌ No hay imágenes fuente con formato IMG-[FECHA]-WA[NUM]")
        return

    print(f"⚙️ Aumentando {os.path.basename(class_path)}: {total_real} → {TARGET_COUNT}")

    # --------------------------------------------------
    # 3. FECHA ACTUAL
    # --------------------------------------------------
    today = datetime.now().strftime("%Y%m%d")

    # --------------------------------------------------
    # 4. CALCULAR SIGUIENTE WA GLOBAL (SIN REPETIR)
    # --------------------------------------------------
    wa_numbers = []
    for f in all_images:
        m = re.search(r"-WA(\d+)\.", f)
        if m:
            wa_numbers.append(int(m.group(1)))

    next_wa = max(wa_numbers) + 1 if wa_numbers else 1

    # --------------------------------------------------
    # 5. AUMENTO CONTROLADO (CORTE DURO)
    # --------------------------------------------------
    current_count = total_real

    with tqdm(total=TARGET_COUNT, initial=current_count) as pbar:
        while current_count < TARGET_COUNT:

            img_name = random.choice(source_images)
            img_path = os.path.join(class_path, img_name)

            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            aug_img = augmenter(image=img)["image"]

            out_name = f"IMG-{today}-WA{next_wa:05d}.jpg"
            out_path = os.path.join(class_path, out_name)

            cv2.imwrite(
                out_path,
                cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR)
            )

            next_wa += 1
            current_count += 1
            pbar.update(1)

# ==================================================
# EJECUCIÓN
# ==================================================
print(f"\n🔍 Dataset: {DATASET_DIR}\n")

for cls in CLASSES_TO_AUGMENT:
    cls_dir = os.path.join(DATASET_DIR, cls)
    if not os.path.isdir(cls_dir):
        print(f"❌ No existe la carpeta: {cls}")
        continue
    augment_class(cls_dir)

print("\n✅ AUMENTO COMPLETADO (META EXACTA, FORMATO UNIFICADO)")


In [ ]:
import os
import cv2
import random
import albumentations as A
from tqdm import tqdm

# ==================================================
# CONFIGURACIÓN GENERAL
# ==================================================
DATASET_DIR = "imagenes_maiz"
TARGET_COUNT = 650
IMG_EXTS = (".jpg", ".jpeg", ".png")

CLASSES_TO_AUGMENT = ["tizon_foliar"]

# ==================================================
# PIPELINE DE AUMENTO DEFINITIVO (SIN CONFUSIÓN)
# ==================================================
augmenter = A.Compose([

    # ==================================================
    # GEOMÉTRICOS
    # ==================================================
    A.Rotate(
        limit=5,
        border_mode=cv2.BORDER_CONSTANT,
        value=(0, 0, 0),
        p=0.4
    ),

    A.HorizontalFlip(p=0.3),

    A.RandomResizedCrop(
        size=(224, 224),
        scale=(0.92, 1.0),
        ratio=(0.97, 1.03),
        p=0.35
    ),

    A.PadIfNeeded(
        min_height=224,
        min_width=224,
        border_mode=cv2.BORDER_CONSTANT,
        value=(0, 0, 0),
        p=0.15
    ),

    # ==================================================
    # COLOR / ILUMINACIÓN
    # ==================================================
    A.RandomBrightnessContrast(
        brightness_limit=0.07,
        contrast_limit=0.07,
        p=0.3
    ),

    A.HueSaturationValue(
        hue_shift_limit=8,
        sat_shift_limit=10,
        val_shift_limit=8,
        p=0.3
    ),

    A.RGBShift(
        r_shift_limit=8,
        g_shift_limit=8,
        b_shift_limit=8,
        p=0.25
    ),

    A.ChannelShuffle(p=0.15),

    A.CLAHE(
        clip_limit=2.0,
        tile_grid_size=(8, 8),
        p=0.25
    ),

    # ==================================================
    # RUIDO / BLUR
    # ==================================================
    A.GaussNoise(
        var_limit=(5, 15),
        p=0.25
    ),

    A.Blur(
        blur_limit=3,
        p=0.2
    ),

    A.GaussianBlur(
        blur_limit=3,
        p=0.2
    ),

    # ==================================================
    # DISTORSIONES (SUAVES)
    # ==================================================
    A.GridDistortion(
        num_steps=5,
        distort_limit=0.03,
        p=0.15
    ),

    A.OpticalDistortion(
        distort_limit=0.03,
        shift_limit=0.03,
        p=0.15
    ),

    A.ElasticTransform(
        alpha=1,
        sigma=20,
        alpha_affine=5,
        p=0.1
    ),
])


# ==================================================
# FUNCIÓN DE AUMENTO POR CLASE
# ==================================================
def augment_class(class_path):
    images = [f for f in os.listdir(class_path) if f.lower().endswith(IMG_EXTS)]
    current = len(images)

    if current >= TARGET_COUNT:
        print(f"✅ {os.path.basename(class_path)} ya tiene {current} imágenes")
        return

    needed = TARGET_COUNT - current
    print(f"⚙️ Aumentando {os.path.basename(class_path)}: {current} → {TARGET_COUNT}")

    for i in tqdm(range(needed)):
        img_name = random.choice(images)
        img_path = os.path.join(class_path, img_name)

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        aug_img = augmenter(image=img)["image"]

        out_name = f"aug_{current+i:04d}.jpg"
        out_path = os.path.join(class_path, out_name)

        cv2.imwrite(out_path, cv2.cvtColor(aug_img, cv2.COLOR_RGB2BGR))

# ==================================================
# EJECUCIÓN CONTROLADA
# ==================================================
print(f"\n🔍 Dataset: {DATASET_DIR}\n")

for cls in CLASSES_TO_AUGMENT:
    cls_dir = os.path.join(DATASET_DIR, cls)
    if not os.path.isdir(cls_dir):
        print(f"❌ No existe la carpeta: {cls}")
        continue
    augment_class(cls_dir)

print("\n✅ AUMENTO COMPLETADO (PIPELINE CORREGIDO Y PROPORCIONAL)")


In [ ]:
# @title CELDA 5: División 70/15/15 y Creación de tf.data (Dataset Balanceado)
import os
import glob
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

# --- 1. CONFIGURACIÓN ---
# Usamos explícitamente la carpeta que acabamos de crear en la Celda 4
DATASET_PATH = 'imagenes_dataset_final'
IMG_SIZE = 224 # Tamaño estándar para EfficientNet/ResNet
BATCH_SIZE = 32

print(f"📂 Cargando imágenes desde: {DATASET_PATH}")

# --- 2. CARGAR RUTAS Y ETIQUETAS ---
try:
    # Obtener nombres de clases ordenados alfabéticamente
    class_names = sorted([d for d in os.listdir(DATASET_PATH) 
                          if os.path.isdir(os.path.join(DATASET_PATH, d))])
    num_classes = len(class_names)
    print(f"✅ Clases detectadas ({num_classes}): {class_names}")

    filepaths = []
    labels = []

    for idx, cls in enumerate(class_names):
        class_dir = os.path.join(DATASET_PATH, cls)
        # Buscar imágenes recursivamente
        for root, dirs, files in os.walk(class_dir):
            for f in files:
                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
                    filepaths.append(os.path.join(root, f))
                    labels.append(idx)
    
    print(f"📊 Total de imágenes encontradas: {len(filepaths)}")

except FileNotFoundError:
    print(f"❌ ERROR: No existe la carpeta '{DATASET_PATH}'. Ejecuta la Celda 4 primero.")
    raise

# --- 3. DIVISIÓN ESTRATIFICADA (70% Train - 15% Val - 15% Test) ---
# Convertir a numpy para facilitar manejo
filepaths = np.array(filepaths)
labels = np.array(labels)

# PASO A: Separar el 15% para TEST (El resto es 85%)
X_temp, X_test, y_temp, y_test = train_test_split(
    filepaths, labels, 
    test_size=0.15, 
    stratify=labels, 
    random_state=42
)

# PASO B: Separar el 15% (del total original) para VALIDACIÓN
# Como nos queda el 85% de los datos, calculamos: 15 / 85 = 0.1765
val_split = 0.15 / 0.85 
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, 
    test_size=val_split, 
    stratify=y_temp, 
    random_state=42
)

print("-" * 50)
print(f"TRAIN (Entrenamiento 70%): {len(X_train)} imágenes")
print(f"VAL   (Validación 15%):    {len(X_val)} imágenes")
print(f"TEST  (Prueba 15%):        {len(X_test)} imágenes")
print("-" * 50)

# --- 4. CONSTRUCCIÓN DE TF.DATA (Optimización) ---

def process_path(file_path, label):
    """Carga, decodifica y redimensiona la imagen"""
    # Cargar archivo
    img = tf.io.read_file(file_path)
    # Decodificar (detecta jpg o png automáticamente)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    # Convertir a float32 y normalizar [0,1]
    img = tf.cast(img, tf.float32)
    # Redimensionar (EfficientNet necesita tamaño fijo)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return img, label

def configure_for_performance(ds, is_train=False):
    """Optimiza el pipeline de carga"""
    # Cachear en memoria si cabe, sino en disco
    # ds = ds.cache() 
    if is_train:
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

# Crear los Datasets de TensorFlow
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Mapear la función de procesamiento
train_ds = train_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)

# Optimizar rendimiento
train_dataset = configure_for_performance(train_ds, is_train=True)
val_dataset = configure_for_performance(val_ds)
test_dataset = configure_for_performance(test_ds)

print("✅ Pipelines de tf.data creados exitosamente.")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Image Size: {IMG_SIZE}x{IMG_SIZE}")

In [1]:
# ============================================================
# CELDA 5: Carga de datasets desde carpetas train/val/test
# ============================================================

import os
import tensorflow as tf

# ------------------------------------------------------------
# CONFIGURACIÓN
# ------------------------------------------------------------

BASE_PATH = "/mnt/d/Deteccion de enfermedades en maiz/imagenes_dataset_final"

TRAIN_DIR = os.path.join(BASE_PATH, "train")
VAL_DIR   = os.path.join(BASE_PATH, "val")
TEST_DIR  = os.path.join(BASE_PATH, "test")

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

print("📂 Directorios configurados:")
print("TRAIN:", TRAIN_DIR)
print("VAL  :", VAL_DIR)
print("TEST :", TEST_DIR)

# ------------------------------------------------------------
# CARGA DE DATASETS
# ------------------------------------------------------------

train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="int",
    color_mode="rgb",
    image_size=(IMG_SIZE, IMG_SIZE),
    interpolation="bilinear",
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="int",
    color_mode="rgb",
    image_size=(IMG_SIZE, IMG_SIZE),
    interpolation="bilinear",
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels="inferred",
    label_mode="int",
    color_mode="rgb",
    image_size=(IMG_SIZE, IMG_SIZE),
    interpolation="bilinear",
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ------------------------------------------------------------
# NOMBRES DE CLASES
# ------------------------------------------------------------

class_names = train_dataset.class_names
num_classes = len(class_names)

print("-" * 50)
print(f"✅ Clases detectadas ({num_classes}):")
print(class_names)
print("-" * 50)

# ------------------------------------------------------------
# PREPROCESAMIENTO
# ------------------------------------------------------------
# Sin normalización manual: las imágenes quedan en [0, 255] (float32).
# EfficientNetB0 incluye su propia normalización interna.
# Para DenseNet121 aplicar preprocess_input (ver abajo).

# ------------------------------------------------------------
# OPTIMIZACIÓN DEL PIPELINE
# ------------------------------------------------------------

train_dataset = train_dataset.cache().prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(tf.data.AUTOTUNE)

print("✅ Pipelines tf.data creados correctamente.")
print(f"📏 Tamaño de imagen : {IMG_SIZE}x{IMG_SIZE}")
print(f"📦 Batch size       : {BATCH_SIZE}")

I0000 00:00:1792280562.328321   23960 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792280562.607832   23960 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792280612.529621   23960 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


📂 Directorios configurados:
TRAIN: /mnt/d/Deteccion de enfermedades en maiz/imagenes_dataset_final/train
VAL  : /mnt/d/Deteccion de enfermedades en maiz/imagenes_dataset_final/val
TEST : /mnt/d/Deteccion de enfermedades en maiz/imagenes_dataset_final/test
Found 1950 files belonging to 3 classes.
Found 299 files belonging to 3 classes.


W0000 00:00:1792280656.859834   23960 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Found 301 files belonging to 3 classes.
--------------------------------------------------
✅ Clases detectadas (3):
['roya_comun', 'sana', 'tizon_foliar']
--------------------------------------------------
✅ Pipelines tf.data creados correctamente.
📏 Tamaño de imagen : 224x224
📦 Batch size       : 32


In [ ]:
import os

TRAIN_DIR = "/mnt/d/Deteccion de enfermedades en maiz/imagenes_dataset_final/train"

class_names = sorted([
    carpeta for carpeta in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, carpeta))
])

num_classes = len(class_names)

print(f"Número de clases: {num_classes}")
print(f"Nombres de clases: {class_names}")

In [2]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU disponibles:", tf.config.list_physical_devices("GPU"))
print("Compilado con CUDA:", tf.test.is_built_with_cuda())
print("Información de compilación:", tf.sysconfig.get_build_info())

TensorFlow: 2.21.0
GPU disponibles: []
Compilado con CUDA: True
Información de compilación: OrderedDict({'cpu_compiler': 'clang', 'cuda_compute_capabilities': ['sm_60', 'sm_70', 'sm_80', 'sm_89', 'compute_90'], 'cuda_version': '12.5.1', 'cudnn_version': '9', 'is_cuda_build': True, 'is_rocm_build': False, 'is_tensorrt_build': False})


In [ ]:
import tensorflow as tf
print("¿Hay GPU disponible?: ", tf.config.list_physical_devices('GPU'))

In [ ]:
import tensorflow as tf
print("Versión de TF:", tf.__version__)

In [2]:
"""Entrenamiento de DenseNet121 (base congelada) para clasificación de hojas de maíz.

Entradas esperadas (definidas en el pipeline de datos):
    train_dataset, val_dataset, test_dataset: tf.data.Dataset de (imagen, etiqueta entera)
    - Imágenes float32 de 224 x 224 x 3 con valores en [0, 255] (sin dividir entre 255).
      El modelo aplica internamente el preprocesamiento de DenseNet121 (escala a [0, 1] y
      normalización con la media y desviación de ImageNet, equivalente a
      tf.keras.applications.densenet.preprocess_input), de modo que la aplicación web
      también debe enviar las imágenes en [0, 255].
    class_names: lista de clases en el orden de las etiquetas (train_dataset.class_names,
      obtenida ANTES de aplicar .map() o .cache() sobre el dataset).
"""

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import applications, callbacks, layers, models

# ==================================================
# 1. CONFIGURACIÓN
# ==================================================

SEED = 42

IMG_SIZE = 224
INPUT_SHAPE = (IMG_SIZE, IMG_SIZE, 3)

LEARNING_RATE = 1e-3
DROPOUT_RATE = 0.3
EPOCHS = 100

ES_PATIENCE = 10        # EarlyStopping
LR_PATIENCE = 5         # ReduceLROnPlateau
LR_FACTOR = 0.2
MIN_LR = 1e-6

# Estadísticas de ImageNet usadas por DenseNet121 (modo "torch" de preprocess_input)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

OUTPUT_DIR = Path("salidas_densenet121")


# ==================================================
# 2. FUNCIONES
# ==================================================

def verificar_rango_entrada(dataset: tf.data.Dataset) -> None:
    """Verifica que las imágenes lleguen en [0, 255] y no en [0, 1]."""
    imgs, _ = next(iter(dataset))
    max_pixel = float(tf.reduce_max(imgs))
    min_pixel = float(tf.reduce_min(imgs))
    if min_pixel < 0 or max_pixel <= 1.5:
        raise ValueError(
            f"Rango de píxeles inesperado: [{min_pixel:.2f}, {max_pixel:.2f}]. "
            "El modelo DenseNet121 espera valores de 0 a 255 y aplica su preprocesamiento "
            "internamente; elimine la división entre 255 en el pipeline de datos."
        )


def construir_modelo(num_classes: int) -> tf.keras.Model:
    """DenseNet121 preentrenada y congelada, con preprocesamiento y cabeza de clasificación."""
    base_model = applications.DenseNet121(
        include_top=False,
        weights="imagenet",
        input_shape=INPUT_SHAPE,
    )
    base_model.trainable = False

    inputs = layers.Input(shape=INPUT_SHAPE)
    # Preprocesamiento de DenseNet121: x / 255 y normalización con media y desviación de ImageNet
    x = layers.Rescaling(1.0 / 255)(inputs)
    x = layers.Normalization(mean=IMAGENET_MEAN, variance=[s ** 2 for s in IMAGENET_STD])(x)

    x = base_model(x, training=False)        # mantiene BatchNorm en modo inferencia
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs, name="DenseNet121_Maiz")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def construir_callbacks(checkpoint_path: Path) -> list:
    """Callbacks que monitorean la misma métrica (val_loss)."""
    return [
        callbacks.ModelCheckpoint(
            filepath=str(checkpoint_path),
            save_best_only=True,
            monitor="val_loss",
            mode="min",
            verbose=1,
        ),
        callbacks.EarlyStopping(
            monitor="val_loss",
            patience=ES_PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=LR_FACTOR,
            patience=LR_PATIENCE,
            min_lr=MIN_LR,
            verbose=1,
        ),
    ]


def evaluar(model: tf.keras.Model, dataset: tf.data.Dataset, class_names: list,
            output_dir: Path) -> dict:
    """Evalúa en prueba, guarda el reporte y la matriz de confusión."""
    test_loss, test_acc = model.evaluate(dataset, verbose=0)

    # Etiquetas y predicciones se obtienen en la misma pasada,
    # por lo que quedan alineadas aunque el dataset se baraje.
    y_true, y_pred = [], []
    for x, y in dataset:
        y_true.append(y.numpy())
        y_pred.append(model.predict_on_batch(x).argmax(axis=1))
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    etiquetas = list(range(len(class_names)))
    reporte = classification_report(
        y_true, y_pred, labels=etiquetas, target_names=class_names,
        digits=4, output_dict=True, zero_division=0,
    )
    matriz = confusion_matrix(y_true, y_pred, labels=etiquetas)

    print(f"\nImágenes evaluadas: {len(y_true)}")
    print(f"Accuracy: {test_acc:.4f} | Loss: {test_loss:.4f}")
    print(classification_report(
        y_true, y_pred, labels=etiquetas, target_names=class_names,
        digits=4, zero_division=0,
    ))
    print("Matriz de confusión:")
    print(matriz)

    resultados = {
        "n_imagenes": int(len(y_true)),
        "accuracy": float(test_acc),
        "loss": float(test_loss),
        "reporte": reporte,
        "matriz_confusion": matriz.tolist(),
    }
    with open(output_dir / "resultados_test.json", "w", encoding="utf-8") as f:
        json.dump(resultados, f, ensure_ascii=False, indent=2)

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(matriz, cmap="Blues")
    ax.set_xticks(etiquetas)
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticks(etiquetas)
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
    for i in etiquetas:
        for j in etiquetas:
            ax.text(j, i, matriz[i, j], ha="center", va="center")
    fig.tight_layout()
    fig.savefig(output_dir / "matriz_confusion.png", dpi=300)
    plt.close(fig)

    return resultados


def graficar_historial(history: tf.keras.callbacks.History, output_dir: Path) -> None:
    """Guarda las curvas de exactitud y pérdida."""
    h = history.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(h["accuracy"], label="Train Accuracy")
    axes[0].plot(h["val_accuracy"], label="Validation Accuracy")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(h["loss"], label="Train Loss")
    axes[1].plot(h["val_loss"], label="Validation Loss")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(output_dir / "curvas_entrenamiento.png", dpi=300)
    plt.close(fig)

    with open(output_dir / "historial.json", "w", encoding="utf-8") as f:
        json.dump({k: [float(v) for v in vals] for k, vals in h.items()}, f, indent=2)


# ==================================================
# 3. EJECUCIÓN
# ==================================================

def main(train_dataset, val_dataset, test_dataset, class_names) -> None:
    tf.keras.utils.set_random_seed(SEED)     # reproducibilidad (Python, NumPy, TensorFlow)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # El atributo model.class_names no se conserva al guardar en .keras
    with open(OUTPUT_DIR / "class_names.json", "w", encoding="utf-8") as f:
        json.dump(list(class_names), f, ensure_ascii=False)

    verificar_rango_entrada(train_dataset)

    model = construir_modelo(len(class_names))
    model.summary()

    print("\nIniciando entrenamiento (DenseNet121)...")
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=EPOCHS,
        callbacks=construir_callbacks(OUTPUT_DIR / "Model_DenseNet121.keras"),
    )
    print(f"Épocas ejecutadas: {len(history.history['loss'])}")

    graficar_historial(history, OUTPUT_DIR)

    print("\nEvaluando modelo en test...")
    evaluar(model, test_dataset, class_names, OUTPUT_DIR)

    print("\nEntrenamiento completado.")


if __name__ == "__main__":
    main(train_dataset, val_dataset, test_dataset, class_names)   # definidos en la celda 5

W0000 00:00:1792280685.734180   23960 cache_dataset_ops.cc:912] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


Model: "DenseNet121_Maiz"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalization (Normalization)   │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │         3,075 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,040,579 (26.86 MB)

 Trainable params: 3,075 (12.01 KB)

 Non-trainable params: 7,037,504 (26.85 MB)


Iniciando entrenamiento (DenseNet121)...
Epoch 1/100


/mnt/d/Deteccion de enfermedades en maiz/env_maiz_312/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5923 - loss: 0.9056
Epoch 1: val_loss improved from None to 0.49851, saving model to salidas_densenet121/Model_DenseNet121.keras

Epoch 1: finished saving model to salidas_densenet121/Model_DenseNet121.keras
61/61 ━━━━━━━━━━━━━━━━━━━━ 145s 2s/step - accuracy: 0.5923 - loss: 0.9056 - val_accuracy: 0.8495 - val_loss: 0.4985 - learning_rate: 0.0010
Epoch 2/100
61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7831 - loss: 0.5280
Epoch 2: val_loss improved from 0.49851 to 0.38551, saving model to salidas_densenet121/Model_DenseNet121.keras

Epoch 2: finished saving model to salidas_densenet121/Model_DenseNet121.keras
61/61 ━━━━━━━━━━━━━━━━━━━━ 113s 2s/step - accuracy: 0.7831 - loss: 0.5280 - val_accuracy: 0.8696 - val_loss: 0.3855 - learning_rate: 0.0010
Epoch 3/100
61/61 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8154 - loss: 0.4488
Epoch 3: val_loss improved from 0.38551 to 0.34937, saving model to salidas_densenet121/Model_DenseNe

In [3]:
"""Indicadores de rendimiento computacional (ecuaciones 13, 14 y 15) por imagen.

Ejecutar en un proceso nuevo (terminal), NO dentro del notebook:
    python medir_rendimiento.py
El notebook mantiene en RAM los datasets con .cache() y eso inflaría el consumo de memoria.
"""

import json
import os
import platform
import time

import numpy as np
import psutil
import tensorflow as tf

# ==================================================
# CONFIGURACIÓN
# ==================================================

BASE_PATH = "/mnt/d/Deteccion de enfermedades en maiz/imagenes_dataset_final"
TEST_DIR = os.path.join(BASE_PATH, "test")
MODEL_PATH = os.path.join(BASE_PATH, "salidas_densenet121", "Model_DenseNet121.keras")
OUTPUT_PATH = os.path.join(BASE_PATH, "salidas_densenet121", "indicadores_rendimiento.json")

IMG_SIZE = 224
N_CALENTAMIENTO = 10      # inferencias iniciales que no se miden


# ==================================================
# FUNCIONES
# ==================================================

def describir_equipo() -> dict:
    """Datos del equipo para declarar en el informe."""
    cpu = platform.processor()
    try:
        with open("/proc/cpuinfo", encoding="utf-8") as f:
            cpu = next(l.split(":")[1].strip() for l in f if l.startswith("model name"))
    except (OSError, StopIteration):
        pass
    return {
        "cpu": cpu,
        "nucleos_logicos": psutil.cpu_count(logical=True),
        "ram_total_gb": round(psutil.virtual_memory().total / 1024 ** 3, 2),
        "gpu": [d.name for d in tf.config.list_physical_devices("GPU")] or "ninguna",
        "sistema": platform.platform(),
        "tensorflow": tf.__version__,
    }


def medir_indicadores(predict_fn, imagenes, n_calentamiento: int = N_CALENTAMIENTO) -> dict:
    """Mide TR, CM y CPU por imagen. `imagenes` es un iterable de tensores (1, H, W, 3)."""
    proceso = psutil.Process()
    n_nucleos = psutil.cpu_count(logical=True)
    ram_total = psutil.virtual_memory().total

    tiempos, memoria = [], []
    cpu_activo, cpu_total = 0.0, 0.0

    for i, x in enumerate(imagenes):
        if i < n_calentamiento:                       # calentamiento (no se mide)
            predict_fn(x)
            continue

        c0 = proceso.cpu_times()
        t0 = time.perf_counter()                      # Ti
        predict_fn(x)
        t1 = time.perf_counter()                      # Tf
        c1 = proceso.cpu_times()

        tiempos.append(t1 - t0)                       # Ec. 13: TR = Tf - Ti
        memoria.append(proceso.memory_info().rss)     # Ec. 14: RAM utilizada
        cpu_activo += (c1.user + c1.system) - (c0.user + c0.system)
        cpu_total += t1 - t0

    tiempos = np.array(tiempos)
    memoria = np.array(memoria)
    cpu_por_nucleo = cpu_activo / cpu_total * 100     # Ec. 15 (puede superar 100 con varios hilos)

    return {
        "n_imagenes_medidas": int(len(tiempos)),
        "TR_media_s": float(tiempos.mean()),
        "TR_desv_s": float(tiempos.std(ddof=1)),
        "TR_p95_s": float(np.percentile(tiempos, 95)),
        "CM_media_percent": float(memoria.mean() / ram_total * 100),
        "RAM_media_MB": float(memoria.mean() / 1024 ** 2),
        "CPU_percent_capacidad_total": float(cpu_por_nucleo / n_nucleos),
        "CPU_percent_por_nucleo": float(cpu_por_nucleo),
    }


# ==================================================
# EJECUCIÓN
# ==================================================

if __name__ == "__main__":
    model = tf.keras.models.load_model(MODEL_PATH)

    # Una imagen por inferencia, leída desde disco sin cache
    imagenes = (
        x for x, _ in tf.keras.utils.image_dataset_from_directory(
            TEST_DIR, labels="inferred", label_mode="int",
            image_size=(IMG_SIZE, IMG_SIZE), batch_size=1, shuffle=False,
        )
    )

    indicadores = medir_indicadores(lambda x: model.predict(x, verbose=0), imagenes)
    resultado = {"equipo": describir_equipo(), "EfficientNetB0": indicadores}

    print("\n===== INDICADORES DE RENDIMIENTO (por imagen) =====")
    print(f"Imágenes medidas        : {indicadores['n_imagenes_medidas']}")
    print(f"Tiempo de respuesta (TR): {indicadores['TR_media_s']:.4f} +/- "
          f"{indicadores['TR_desv_s']:.4f} s (p95: {indicadores['TR_p95_s']:.4f} s)")
    print(f"Consumo de memoria (CM) : {indicadores['CM_media_percent']:.2f} % "
          f"({indicadores['RAM_media_MB']:.0f} MB)")
    print(f"Uso de CPU              : {indicadores['CPU_percent_capacidad_total']:.2f} % "
          f"de la capacidad total ({indicadores['CPU_percent_por_nucleo']:.0f} % por núcleo)")

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(resultado, f, ensure_ascii=False, indent=2)
    print(f"\nGuardado en: {OUTPUT_PATH}")

Found 301 files belonging to 3 classes.

===== INDICADORES DE RENDIMIENTO (por imagen) =====
Imágenes medidas        : 291
Tiempo de respuesta (TR): 0.1693 +/- 0.0247 s (p95: 0.2076 s)
Consumo de memoria (CM) : 24.95 % (3964 MB)
Uso de CPU              : 25.86 % de la capacidad total (207 % por núcleo)

Guardado en: /mnt/d/Deteccion de enfermedades en maiz/imagenes_dataset_final/salidas_densenet121/indicadores_rendimiento.json


In [ ]:
# @title CELDA 7: Evaluación de Métricas y Matriz de Confusión
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# --- 1. OBTENER PREDICCIONES Y ETIQUETAS REALES ---
print("🔍 Generando predicciones sobre el conjunto de prueba...")

# Extraemos las etiquetas verdaderas (y_true) y las predicciones (y_pred)
y_true = []
y_pred_probs = []

# Iteramos sobre el dataset de prueba (batch por batch para no saturar memoria)
for images, labels in test_dataset:
    # Predicción del modelo (probabilidades)
    probs = model.predict(images, verbose=0)
    
    # Guardamos resultados
    y_true.extend(labels.numpy())
    y_pred_probs.extend(probs)

# Convertimos a arrays de numpy
y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)
# Convertimos probabilidades a etiquetas (0, 1, 2, 3) tomando el valor más alto
y_pred = np.argmax(y_pred_probs, axis=1)

print("✅ Predicciones completadas.")
print("-" * 60)

# --- 2. REPORTE DE CLASIFICACIÓN (TABLA DE MÉTRICAS) ---
print("\n📊 REPORTE DE CLASIFICACIÓN (Métricas Oficiales):\n")

report = classification_report(
    y_true, 
    y_pred, 
    target_names=class_names, # Usa los nombres de tus carpetas
    digits=4 # 4 decimales para mayor precisión
)
print(report)

# --- 3. MATRIZ DE CONFUSIÓN (GRÁFICO) ---
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, 
    annot=True,         # Mostrar números dentro de los cuadros
    fmt='d',            # Formato entero (sin decimales)
    cmap='Blues',       # Color azul profesional
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title('Matriz de Confusión - EfficientNetB0', fontsize=16)
plt.ylabel('Etiqueta Real (Verdad)', fontsize=12)
plt.xlabel('Predicción del Modelo', fontsize=12)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# --- 4. EXPLICACIÓN RÁPIDA ---
acc = np.sum(y_pred == y_true) / len(y_true)
print("-" * 60)
print(f"🏆 EXACTITUD GLOBAL (Accuracy): {acc:.2%}")
print("-" * 60)

In [ ]:
# ==================================================
# CELDA 8: Cargar modelo entrenado EfficientNetB0
# ==================================================
import tensorflow as tf

model_path = "Model_EfficientNetB0.keras"

print("📦 Cargando modelo entrenado...")

model = tf.keras.models.load_model(model_path)

print("✅ Modelo cargado correctamente.")
model.summary()


In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image

# ==================================================
# 1. CARGAR MODELO ENTRENADO
# ==================================================

MODEL_PATH = "Model_EfficientNetB0.keras"

model = tf.keras.models.load_model(MODEL_PATH)

print("✅ Modelo cargado correctamente")

# ==================================================
# 2. CONFIGURACIÓN DEL DATASET TEST
# ==================================================

TEST_DIR = "/mnt/d/Deteccion de enfermedades en maiz/imagenes_dataset_final/test"

class_names = [
    "roya_comun",
    "sana",
    "tizon_foliar"
]

VALID_EXTS = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
)

# ==================================================
# 3. SELECCIONAR CLASE ALEATORIA
# ==================================================

true_class = random.choice(class_names)

class_path = os.path.join(TEST_DIR, true_class)

# Obtener imágenes válidas
images_list = [

    f for f in os.listdir(class_path)

    if f.lower().endswith(VALID_EXTS)
]

if len(images_list) == 0:
    raise ValueError(
        f"No hay imágenes en: {class_path}"
    )

# Elegir imagen aleatoria
selected_image = random.choice(images_list)

file_path = os.path.join(
    class_path,
    selected_image
)

print("\n📂 Imagen seleccionada:")
print(file_path)

# ==================================================
# 4. PREPROCESAMIENTO
# ==================================================

IMG_SIZE = (224, 224)

img = image.load_img(
    file_path,
    target_size=IMG_SIZE
)

img_array = image.img_to_array(img)

# Normalización
img_array = img_array / 255.0

# Crear batch
img_batch = np.expand_dims(
    img_array,
    axis=0
)

# ==================================================
# 5. PREDICCIÓN
# ==================================================

predictions = model.predict(
    img_batch,
    verbose=0
)

predicted_idx = np.argmax(
    predictions[0]
)

predicted_class = class_names[
    predicted_idx
]

confidence = (
    predictions[0][predicted_idx] * 100
)

# ==================================================
# 6. RESULTADOS
# ==================================================

print("\n" + "-" * 50)
print("🌽 RESULTADO DE PREDICCIÓN")
print("-" * 50)

print(f"Clase REAL      : {true_class}")

print(f"Clase PREDICHA  : {predicted_class}")

print(f"Confianza       : {confidence:.2f}%")

# Verificación
if predicted_class == true_class:
    print("✅ Predicción CORRECTA")
else:
    print("❌ Predicción INCORRECTA")

print("-" * 50)

# ==================================================
# 7. PROBABILIDADES POR CLASE
# ==================================================

print("\n📊 Probabilidades por clase:\n")

for i, prob in enumerate(predictions[0]):

    marker = "◄" if i == predicted_idx else ""

    print(
        f"{class_names[i]:<15}: "
        f"{prob * 100:.2f}% {marker}"
    )

In [ ]:
model.output_shape


model.metadata = {
    "class_names": ["roya_comun", "sana", "tizon_foliar"]
}

model.metadata



In [ ]:
model.save("efficientnetB0.keras")


In [ ]:
# @title GRÁFICA FORMATO TESIS: Letras (a)-(f) debajo de las imágenes
import matplotlib.pyplot as plt
import tensorflow as tf
import os
import random

# --- CONFIGURACIÓN ---
DATA_DIR = 'imagenes_maiz'
TARGET_SIZE = 224
CLASSES = ["sana", "roya_comun", "tizon_foliar"]

display_data = []

print("🔍 Buscando imágenes originales (NO aumentadas)...")

for class_name in CLASSES:
    class_path = os.path.join(DATA_DIR, class_name)
    found_high_res = False
    
    if os.path.isdir(class_path):
        # Listar archivos
        all_files = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        random.shuffle(all_files) 
        
        for fname in all_files:
            # FILTRO 1: Por Nombre (Si empieza con 'aug_', la saltamos)
            if fname.startswith("aug_"):
                continue

            img_path = os.path.join(class_path, fname)
            
            # Cargar imagen
            try:
                img_raw = tf.io.read_file(img_path)
                img_decoded = tf.image.decode_image(img_raw, channels=3, expand_animations=False)
            except:
                continue 

            # FILTRO 2: Por Tamaño (Si ya es pequeña o igual a 224, la saltamos)
            if img_decoded.shape[0] > 300 or img_decoded.shape[1] > 300:
                
                # Crear la versión redimensionada (Preprocesada)
                img_resized = tf.image.resize(img_decoded, [TARGET_SIZE, TARGET_SIZE])
                img_resized = tf.cast(img_resized, tf.uint8)
                
                display_data.append({
                    "original": img_decoded,
                    "resized": img_resized,
                    "class": class_name,
                    "orig_shape": img_decoded.shape,
                })
                found_high_res = True
                print(f"   ✅ {class_name}: OK -> {fname}")
                break 
        
        if not found_high_res:
            print(f"   ⚠️ {class_name}: No se encontró ninguna imagen original grande.")

# --- 2. GRAFICAR CON FORMATO DE TESIS ---
if len(display_data) > 0:
    # Ajustamos el tamaño de la figura para dar espacio a los textos
    fig, axes = plt.subplots(len(display_data), 2, figsize=(10, 5 * len(display_data)))
    
    labels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)']
    
    for i, data in enumerate(display_data):
        class_label = data["class"].replace("_", " ").upper()
        
        # --- Columna Izquierda: Original ---
        ax_orig = axes[i, 0] if len(display_data) > 1 else axes[0]
        ax_orig.imshow(data["original"])
        
        
        # LETRA ABAJO (a, c, e)
        label_char = labels[i*2]
        ax_orig.text(0.5, -0.1, label_char, transform=ax_orig.transAxes, 
                     ha='center', va='top', fontsize=14, weight='bold', color='black')
        
        ax_orig.axis('off')
        
        # --- Columna Derecha: Redimensionada ---
        ax_res = axes[i, 1] if len(display_data) > 1 else axes[1]
        ax_res.imshow(data["resized"])
        
        # LETRA ABAJO (b, d, f)
        label_char = labels[i*2 + 1]
        ax_res.text(0.5, -0.1, label_char, transform=ax_res.transAxes, 
                    ha='center', va='top', fontsize=14, weight='bold', color='black')
        
        ax_res.axis('off')

    # Ajustar espaciado para que no se corten las letras de abajo
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.05) 
    plt.show()
else:
    print("❌ No se encontraron imágenes aptas para mostrar.")

In [ ]:
# ==================================================
# CELDA: Subir imagen y predecir clase (UI simple)
# ==================================================
!pip install ipywidgets


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
import io

# --------------------------------
# CONFIGURACIÓN
# --------------------------------
IMG_SIZE = (224, 224)

# MISMO ORDEN que en el entrenamiento
class_names = ["roya_comun", "sana", "tizon_foliar"]

# --------------------------------
# WIDGET PARA SUBIR IMAGEN
# --------------------------------
uploader = widgets.FileUpload(
    accept='image/*',
    multiple=False
)

display(uploader)

# --------------------------------
# FUNCIÓN DE PREDICCIÓN
# --------------------------------
def predecir_imagen(change):
    if len(uploader.value) == 0:
        return

    # Leer imagen subida
    file_info = list(uploader.value.values())[0]
    image_bytes = file_info['content']
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")

    # Mostrar imagen
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Imagen cargada")
    plt.show()

    # Preprocesamiento
    img_resized = img.resize(IMG_SIZE)
    img_array = np.array(img_resized)
    img_array = np.expand_dims(img_array, axis=0)

    # Predicción
    probs = model.predict(img_array, verbose=0)[0]
    pred_index = np.argmax(probs)
    pred_class = class_names[pred_index]
    pred_conf = probs[pred_index]

    # Resultados
    print("\n📊 RESULTADO DE LA PREDICCIÓN")
    print("-" * 40)

    for cls, p in zip(class_names, probs):
        print(f"{cls:12s}: {p*100:.2f}%")

    print("-" * 40)
    print(f"🏷️ Clase predicha: {pred_class}")
    print(f"🎯 Confianza del modelo: {pred_conf*100:.2f}%")

# --------------------------------
# EVENTO
# --------------------------------
uploader.observe(predecir_imagen, names='value')
